# pyMBE and molecular force fields
 
The Python-based Molecule Builder for ESPResSo (pyMBE) can interface with all-atom and CG force fields and automate the setup of force field parameters in ESPResSo.

This tutorial will focus on the AMBER99-SB and Martini 2.2 force fields.

Select mode:
* `"aa"` for all-atom model
* `"cg"` for coarse-grained model

In [ ]:
mode = "aa"

Install dependencies and patch ParmEd:

```sh
pip install vermouth insane ParmEd ipympl
sed -i "s/raise ParameterError('Not all angle parameters found/pass #raise ParameterError('Not all angle parameters found/" $(python3 -c "import parmed;print(parmed.__file__.rsplit('/', 1)[0] + '/gromacs/gromacstop.py')")
```

## Introduction <a class="anchor" id="introduction"></a>

Import pyMBE library and other important libraries for this tutorial, such as ESPResSo and ParmEd.

In [ ]:
import os
import warnings
import pyMBE
import espressomd
import numpy as np
import matplotlib.pyplot as plt
import parmed.gromacs
parmed.gromacs.GROMACS_TOPDIR = os.path.join(os.environ["GMXDATA"], "top")

Download and prepare peptide:

In [ ]:
if not os.path.isfile("7ETN.pdb"):
    !wget https://files.rcsb.org/download/7ETN.pdb
if not os.path.isfile("martini_v2.2.itp"):
    !wget https://cgmartini-library.s3.ca-central-1.amazonaws.com/1_Downloads/ff_parameters/martini2/particle-definitions/martini_v2.2.itp
if not os.path.isfile("martini_v2.2_aminoacids.itp"):
    !wget https://cgmartini-library.s3.ca-central-1.amazonaws.com/1_Downloads/ff_parameters/martini2/aminoacids/martini_v2.2_aminoacids.itp
!rm -f topol.top
if mode == "aa":
    !rm -f peptide_aa.gro
if mode == "cg":
    !rm -f peptide_cg.gro
!sed "/ HOH A/d; /BARG A/d; /BLYS A/d; /BSER A/d; / B   /d;" 7ETN.pdb > peptide.pdb
if mode == "aa":
    !gmx pdb2gmx -f peptide.pdb -o peptide_aa.gro -ignh -ff amber99sb -water tip3p
if mode == "cg":
    !martinize2 -f peptide.pdb -x peptide_cg.pdb -o topol.top -ff martini22 -ss C -noscfix
    !gmx editconf -f peptide_cg.pdb -o peptide_cg.gro
    with open("martini.itp", "w") as f:
        f.write('#include "martini_v2.2.itp"\n#include "martini_v2.2_aminoacids.itp"\n')

In [ ]:
def get_coord(atom):
    return [atom.xx, atom.xy, atom.xz]

def get_name(atom):
    if mode == "cg":
        return f"{atom.name}({atom.type})"
    if mode == "aa":
        return atom.name

In [ ]:
with warnings.catch_warnings(action="ignore"):
    gmx_top = parmed.gromacs.GromacsTopologyFile("topol.top")
    gmx_gro = parmed.gromacs.GromacsGroFile.parse(f"peptide_{mode}.gro")
gmx_top.positions = gmx_gro.positions

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
for bond in gmx_top.bonds:
    ax.plot(*np.array([get_coord(bond.atom1), get_coord(bond.atom2)]).T, color="k")
for bond in gmx_top.angles:
    ax.plot(*np.array([get_coord(bond.atom1), get_coord(bond.atom2), get_coord(bond.atom3)]).T, color="k", ls="--")
for atom in gmx_top.atoms:
    ax.scatter(*get_coord(atom), marker="o", s=80, depthshade=False,
               edgecolors=None, c={1: "grey", 6: "green", 7: "blue", 8: "r"}.get(atom.atomic_number, "k"))
for atom in gmx_top.atoms:
    ax.text(*(get_coord(atom) + np.ones(3) * 0.15), get_name(atom))
ax.set_aspect("equal")
ax.set_xlabel("X axis")
ax.set_ylabel("Y axis")
ax.set_zlabel("Z axis")
ax.set_axis_off()
if mode == "aa":
    ax.view_init(elev=-10, azim=165, roll=-155, vertical_axis='z')
if mode == "cg":
    ax.view_init(elev=-12, azim=21, roll=16, vertical_axis='z')
plt.tight_layout()

Let us now create an instance of the ESPResSo system where we will place our molecules (a square simulation box with length = `box_l`).

In [ ]:
pmb = pyMBE.pymbe_library(seed=42)

When pyMBE is inicialized, a default system of reduced units is defined. 

* Unit_length = 0.355 nm.
* Unit_charge = 1 elementary charge.
* Temperature = 298.15 K.

The active set of reduced units can be consulted using:

In [ ]:
print(pmb.get_reduced_units())

In [ ]:
Box_L = 7.5*pmb.units.nm

espresso_system = espressomd.System(box_l = [Box_L.to('reduced_length').magnitude]*3)

print('The side of the simulation box is ', Box_L, '=' ,Box_L.to('reduced_length'))


## How to create simple polymers <a class="anchor" id="simple_polymers"></a>

pyMBE can be used to easily construct coarse-grained models of simple polymers. Let us consider a coarse grained model for polydehydroalanaline (PDha) (figure below) in which its monomeric unit can be represented by three beads, as depicted in the schematics below: a backbone bead (grey), a bead for the carboxylic acid group (red) and a bead for the amino group (blue).

<img src="../figs/PDha.png" width=150 height=150 />


To set up such polymer with pyMBE first one has to define the different particles in the monomer (or residue).
Then, one defines the structure of the residue of the polymer. A residue is composed by a `central_bead` where one or various `side_chains` are attached. Each side chain can contain one particle or other residues.

NOTE: All input variables will be given to ESPResSo using these reduced units, since it is a convenient choice for the simulation setup. Internally, pyMBE uses Pint library to deal with unit transformations, which in turn should be  used by the user to define its own variables.

In [ ]:
known_residues = set()
known_atoms = set()
for res in gmx_top.residues:
    if res.name in known_residues:
        continue
    known_residues.add(res.name)
    if mode == "cg":
        central_bead_cg = [get_name(atom) for atom in res.atoms if atom.name == "BB"][0]
    central_bead = "CA" if mode == "aa" else central_bead_cg
    pmb.define_residue(
        name=res.name, 
        central_bead=central_bead,
        side_chains=[get_name(atom) for atom in res.atoms if atom.name not in ["CA", "BB"]])
    for atom in res.atoms:
        if get_name(atom) in known_atoms:
            continue
        known_atoms.add(get_name(atom))
        pmb.define_particle(name=get_name(atom), 
                            z=atom.charge if mode == "cg" else int(1000*atom.charge),
                            sigma=(atom.sigma if mode == "aa" else 1.) * pmb.units('reduced_length'), 
                            epsilon=(atom.epsilon if mode == "aa" else 1.) * pmb.units('reduced_energy'))

In [ ]:
pmb.filter_df(pmb_type='residue')

In [ ]:
pmb.filter_df(pmb_type='particle')

Once done, one has to define a bond for each different type of bond in the polymer. For simplicity, in this tutorial we assume that all bonds are equal and we set-up all bonds using a harmonic potential with the following arbitrary parameters.

In [ ]:
summary = set()
for bond in gmx_top.bonds:
    identifier = f"{get_name(bond.atom1)}, {get_name(bond.atom2)}, {bond.type.k}, {bond.type.req}"
    if identifier in summary:
        continue
    summary.add(identifier)
    pmb.define_bond(bond_type="harmonic",
                    bond_parameters={"k": bond.type.k * pmb.units('reduced_energy / reduced_length**2'),
                                     "r_0": bond.type.req * pmb.units('reduced_length')},
                    particle_pairs = [[get_name(bond.atom1), get_name(bond.atom2)]])

summary = set()
for bond in gmx_top.angles:
    if bond.type is None:
        summary.add(f"need to add bond angle {get_name(bond.atom1)}-{get_name(bond.atom2)}-{get_name(bond.atom3)} theta_0=? k=?")
    else:
        summary.add(f"need to add bond angle {get_name(bond.atom1)}-{get_name(bond.atom2)}-{get_name(bond.atom3)} theta_0={bond.type.theteq:.2f} k={bond.type.k:.2f}")
for x in set(summary):
    print(x)

pmb.add_bonds_to_espresso(espresso_system=espresso_system)

In [ ]:
pmb.filter_df(pmb_type='bond')

NOTE: Currently, only harmonic and FENE bonds are supported.